# Challenge 2: Make the Quantum Coin Biased
**IBM Quantum - Vishnu Institute of Technology | Qiskit Fall Fest 2026**

An H gate gives a fair quantum coin (50% / 50%). In this notebook I use a rotation gate to make the coin biased, so 0 and 1 do not come out equally often.

## Step 1: Importing the libraries

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

sim = AerSimulator()
shots = 100
import numpy as np

## Step 2: Main circuit - biased towards 0
I use an RY rotation gate. RY(theta) turns |0> into cos(theta/2)|0> + sin(theta/2)|1>, so

P(0) = cos^2(theta/2) and P(1) = sin^2(theta/2)

I choose theta = pi/3 (60 degrees). Then P(0) = cos^2(30 degrees) = 0.75 and P(1) = sin^2(30 degrees) = 0.25.

**My prediction:** `0` appears more often, about 75% of the time (around 75 shots), and `1` about 25% (around 25 shots).

This circuit uses only 1 gate, which is within the limit of 4 gates.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.ry(np.pi / 3, 0)
qc.measure(0, 0)
qc.draw('mpl')

In [ ]:
counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
print("Results:", counts)
plot_histogram(counts, title="Biased coin: RY(pi/3)")

## Step 3: Different degrees of bias
If I change the angle, the bias changes. I will try several angles and compare them with the predicted P(1).

In [ ]:
angles = [0, np.pi / 6, np.pi / 3, np.pi / 2, 2 * np.pi / 3, 5 * np.pi / 6, np.pi]
names = ['0', 'pi/6', 'pi/3', 'pi/2', '2pi/3', '5pi/6', 'pi']

all_counts = []
for a, n in zip(angles, names):
    q = QuantumCircuit(1, 1)
    q.ry(a, 0)
    q.measure(0, 0)
    c = sim.run(transpile(q, sim), shots=shots).result().get_counts()
    all_counts.append(c)
    predicted = np.sin(a / 2) ** 2 * 100
    print("theta =", n, "| predicted P(1) = %.1f%%" % predicted, "| result:", c)

In [ ]:
plot_histogram(all_counts, legend=names, title="Different angles", figsize=(12, 5))

## Bonus: Two circuits, one favouring 0 and one favouring 1

**Circuit 1 (0 more frequent):** RY(pi/3). Prediction: about 75% zeros.

**Circuit 2 (1 more frequent):** RY(2pi/3). Prediction: P(1) = sin^2(60 degrees) = 0.75, so about 75% ones.

In [ ]:
qc_zero = QuantumCircuit(1, 1)
qc_zero.ry(np.pi / 3, 0)
qc_zero.measure(0, 0)

qc_one = QuantumCircuit(1, 1)
qc_one.ry(2 * np.pi / 3, 0)
qc_one.measure(0, 0)

counts_zero = sim.run(transpile(qc_zero, sim), shots=shots).result().get_counts()
counts_one = sim.run(transpile(qc_one, sim), shots=shots).result().get_counts()

print("0 favoured:", counts_zero)
print("1 favoured:", counts_one)
plot_histogram([counts_zero, counts_one], legend=['RY(pi/3)', 'RY(2pi/3)'], title="Comparison of the two biased coins")

## Comparing prediction and result
- RY(pi/3) predicted about 75% zeros and 25% ones. The result was close to this, with small differences because only 100 shots are used.
- RY(2pi/3) predicted about 25% zeros and 75% ones. The result was close to this too.
- When the angle was changed, the bias changed. Angle 0 gives all 0, angle pi/2 gives a fair coin (same as H) and angle pi gives all 1.

The two bonus histograms look like mirror images of each other.

## Challenge Question
**How can quantum gates change the probability of obtaining 0 or 1 when a qubit is measured?**

**Answer:** A qubit state is a mix of |0> and |1> with amplitudes. When we measure, the probability of getting 0 or 1 is the square of its amplitude. Quantum gates change these amplitudes. The X gate swaps them, so 0 becomes 1. The H gate makes the amplitudes equal, which gives 50% / 50%. A rotation gate like RY(theta) changes the amplitudes by any amount we choose, so P(0) = cos^2(theta/2) and P(1) = sin^2(theta/2). In this way, by choosing the gates and the angle, we can control how likely 0 or 1 is at measurement.